In [ ]:
import nltk
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import movie_reviews, stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

import warnings
warnings.filterwarnings("ignore")
print("Libraries imported successfully!")

nltk.download("movie_reviews")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
print("\nNLTK datasets downloaded successfully!")

documents = []
for fileid in movie_reviews.fileids():
    review_text = movie_reviews.raw(fileid)
    sentiment = movie_reviews.categories(fileid)[0]
    documents.append({
        "review": review_text,
        "sentiment": sentiment
    })

df = pd.DataFrame(documents)

print("\nDataset loaded successfully!")
print("\nFirst 5 records:")
print(df.head())
print("\nDataset shape:")
print(df.shape)
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

stop_words = set(stopwords.words("english"))
stop_words = stop_words - {"not", "no", "nor"}
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    words = re.findall(r"\b[a-z]+\b", text)
    cleaned_words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]
    return " ".join(cleaned_words)

df["cleaned_review"] = df["review"].apply(clean_text)
print("\nText preprocessing completed!")
print("\nOriginal Review:")
print(df["review"].iloc[0][:300])
print("\nCleaned Review:")
print(df["cleaned_review"].iloc[0][:300])

df["label"] = df["sentiment"].map({
    "neg": 0,
    "pos": 1
})
X = df["cleaned_review"]
y = df["label"]
print("\nTarget labels:")
print("Negative = 0")
print("Positive = 1")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))

pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=10000,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        MultinomialNB()
    )
])

print("\nTF-IDF and Naive Bayes pipeline created!")

parameters = {
    "classifier__alpha": [0.1, 0.5, 1.0, 2.0]
}

grid = GridSearchCV(
    pipeline,
    parameters,
    cv=3,
    scoring="f1",
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("\nBest Parameters:")
print(grid.best_params_)
print("\nBest Cross Validation F1 Score:")
print(round(grid.best_score_, 4))

y_pred = best_model.predict(X_test)

y_prob = best_model.predict_proba(X_test)[:, 1]
print("\nPrediction completed!")

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_prob)

print("\n========== MODEL PERFORMANCE ==========")

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("ROC-AUC  :", round(roc_auc, 4))


# STEP 11: CLASSIFICATION REPORT

print("\n========== CLASSIFICATION REPORT ==========")

print(classification_report(
    y_test,
    y_pred,
    target_names=["Negative", "Positive"]
))


# STEP 12: CONFUSION MATRIX

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)

plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")
plt.title("Sentiment Classification Confusion Matrix")

plt.tight_layout()
plt.savefig("sentiment_confusion_matrix.png")
plt.show()


# STEP 13: ROC CURVE

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))

plt.plot(
    fpr,
    tpr,
    label=f"Naive Bayes (AUC = {roc_auc:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    "k--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Sentiment Classification")

plt.legend()
plt.grid()

plt.tight_layout()
plt.savefig("sentiment_roc_curve.png")
plt.show()


# STEP 14: SAVE DATASET AND RESULTS

df.to_csv(
    "sentiment_dataset.csv",
    index=False
)

results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ]
})

results.to_csv(
    "sentiment_model_results.csv",
    index=False
)

print("\nDataset and model results saved!")


# STEP 15: TEST WITH NEW REVIEWS

def predict_sentiment(review):

    cleaned = clean_text(review)

    prediction = best_model.predict([cleaned])[0]

    probability = best_model.predict_proba([cleaned])[0]

    sentiment = "Positive" if prediction == 1 else "Negative"

    confidence = probability[prediction] * 100

    print("\nReview:", review)
    print("Predicted Sentiment:", sentiment)
    print("Confidence:", round(confidence, 2), "%")


predict_sentiment(
    "This movie was amazing and I really enjoyed it!"
)

predict_sentiment(
    "The movie was boring and a complete waste of time."
)


# STEP 16: FINAL SUMMARY

print("\n======================================")
print("SENTIMENT CLASSIFICATION COMPLETED")
print("======================================")

print("Total Reviews:", len(df))
print("Positive Reviews:", (y == 1).sum())
print("Negative Reviews:", (y == 0).sum())
print("Best Naive Bayes Alpha:", grid.best_params_["classifier__alpha"])
print("Final Accuracy:", round(accuracy * 100, 2), "%")

print("\nGenerated files:")
print("1. sentiment_dataset.csv")
print("2. sentiment_model_results.csv")
print("3. sentiment_confusion_matrix.png")
print("4. sentiment_roc_curve.png")

print("\nPROJECT COMPLETED SUCCESSFULLY!")
